In [1]:
import sys

%cd /kaggle/working
!rm -rf nejm-brain-to-text speech-bci-transformer
!git clone https://github.com/Neuroprosthetics-Lab/nejm-brain-to-text.git
!git -C nejm-brain-to-text checkout 980a5b7
!git clone https://github.com/ritpupp/speech-bci-transformer.git
!cd nejm-brain-to-text && patch -p1 < ../speech-bci-transformer/train/transformer_trainer.patch

# the patched trainer does `from models.transformer import ...` and `from rnn_model import ...`
sys.path.insert(0, "/kaggle/working/speech-bci-transformer")
sys.path.insert(0, "/kaggle/working/nejm-brain-to-text/model_training")

/kaggle/working
Cloning into 'nejm-brain-to-text'...
remote: Enumerating objects: 2977, done.
remote: Counting objects: 100% (255/255), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 2977 (delta 111), reused 105 (delta 105), pack-reused 2722 (from 1)
Receiving objects: 100% (2977/2977), 80.40 MiB | 37.68 MiB/s, done.
Resolving deltas: 100% (403/403), done.
Note: switching to '980a5b7'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 980a5b7 Merge pull request #6 from kakuteki/main
Clonin

In [2]:
!pip install -q pytest
!cd /kaggle/working/speech-bci-transformer && python -m pytest tests -q

.....                                                                    [100%]
5 passed in 2.75s


In [3]:
import rnn_trainer
from models.transformer import TransformerDecoder
print("imports OK")

imports OK


In [4]:
import os, time, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # no CUDA use yet, so this still takes effect

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from omegaconf import OmegaConf
from rnn_trainer import BrainToTextDecoder_Trainer

os.chdir("/kaggle/working/nejm-brain-to-text/model_training")   # rnn_args.yaml is loaded from here
SESSIONS_DIR = "/kaggle/input/competitions/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final"
print(torch.__version__, torch.cuda.get_device_name(0))

QUICK_TEST = True          # True = short test. Set False for the real 12,000-batch run.
COMPILE = False
N_BATCHES = 2000 if QUICK_TEST else 12000
LR_MAX, LR_MIN = 1e-3, 2e-5

RUN_NAME = "tf_quicktest" if QUICK_TEST else f"tf_{N_BATCHES}"

args = OmegaConf.load("rnn_args.yaml")
keys = ["lr_max", "lr_min", "lr_max_day", "lr_min_day", "weight_decay", "weight_decay_day", "epsilon", "grad_norm_clip_value"]
print("official:", {k: args.get(k) for k in keys})

args.dataset.dataset_dir = SESSIONS_DIR
args.dataset.batch_size = 32
args.output_dir = f"/kaggle/working/runs/{RUN_NAME}"
args.checkpoint_dir = f"{args.output_dir}/checkpoint"
args.gpu_number = "0"
args.use_amp = False
args.compile = COMPILE
args.num_training_batches = N_BATCHES
args.lr_decay_steps = args.lr_decay_steps_day = N_BATCHES
args.lr_warmup_steps = args.lr_warmup_steps_day = min(1000, N_BATCHES // 10)
args.batches_per_val_step = 500 if QUICK_TEST else N_BATCHES // 10
args.batches_per_train_log = 100

args.lr_max, args.lr_min = LR_MAX, LR_MIN
args.model.type = "transformer"
args.model.transformer = dict(d_model=384, n_layers=4, n_heads=6, dim_feedforward=1536, dropout=0.1)

shutil.rmtree(args.output_dir, ignore_errors=True)
print("mine:", {k: args.get(k) for k in keys})
print(OmegaConf.to_yaml(args.model.transformer))

2.11.0+cu128 Tesla T4
official: {'lr_max': 0.005, 'lr_min': 0.0001, 'lr_max_day': 0.005, 'lr_min_day': 0.0001, 'weight_decay': 0.001, 'weight_decay_day': 0, 'epsilon': 0.1, 'grad_norm_clip_value': 10}
mine: {'lr_max': 0.001, 'lr_min': 2e-05, 'lr_max_day': 0.005, 'lr_min_day': 0.0001, 'weight_decay': 0.001, 'weight_decay_day': 0, 'epsilon': 0.1, 'grad_norm_clip_value': 10}
d_model: 384
n_layers: 4
n_heads: 6
dim_feedforward: 1536
dropout: 0.1



In [5]:
t0 = time.time()
trainer = BrainToTextDecoder_Trainer(args)
stats = trainer.train()
elapsed = time.time() - t0
print(f"\nWall time: {elapsed / 60:.1f} min for {N_BATCHES} batches ({elapsed / N_BATCHES:.2f} s/batch)")

2026-10-11 21:44:57,027: Using device: cuda:0
2026-10-11 21:44:57,116: Initialized RNN decoding model
2026-10-11 21:44:57,117: TransformerDecoder(
  (day_layer_activation): Softsign()
  (day_weights): ParameterList(
      (0): Parameter containing: [torch.float32 of size 512x512]
      (1): Parameter containing: [torch.float32 of size 512x512]
      (2): Parameter containing: [torch.float32 of size 512x512]
      (3): Parameter containing: [torch.float32 of size 512x512]
      (4): Parameter containing: [torch.float32 of size 512x512]
      (5): Parameter containing: [torch.float32 of size 512x512]
      (6): Parameter containing: [torch.float32 of size 512x512]
      (7): Parameter containing: [torch.float32 of size 512x512]
      (8): Parameter containing: [torch.float32 of size 512x512]
      (9): Parameter containing: [torch.float32 of size 512x512]
      (10): Parameter containing: [torch.float32 of size 512x512]
      (11): Parameter containing: [torch.float32 of size 512x512]
  